In [2]:
import os 

os.chdir('..')
print(os.getcwd())


/Users/bibashnepali/Desktop/AI_Data_Intelligence/ai_data_intelligence


In [3]:
from app.data_loader import DataLoader
from app.cleaner.data_cleaner import DataCleaner
from app.validators.base_validator import BaseValidator
from app.standardizer.base_standardizer import BaseStandardizer

ModuleNotFoundError: No module named 'standardizer'

In [3]:
messy_file_path="/Users/bibashnepali/Desktop/AI_Data_Intelligence/ai_data_intelligence/data/Netflix Dataset.csv"
messy_Data = DataLoader(messy_file_path).load_data()

Data loaded successfully.


In [4]:
print(messy_Data.head())

  Show_Id Category  Title           Director  \
0      s1  TV Show     3%                NaN   
1      s2    Movie  07:19  Jorge Michel Grau   
2      s3    Movie  23:59       Gilbert Chan   
3      s4    Movie      9        Shane Acker   
4      s5    Movie     21     Robert Luketic   

                                                Cast        Country  \
0  João Miguel, Bianca Comparato, Michel Gomes, R...         Brazil   
1  Demián Bichir, Héctor Bonilla, Oscar Serrano, ...         Mexico   
2  Tedd Chan, Stella Chung, Henley Hii, Lawrence ...      Singapore   
3  Elijah Wood, John C. Reilly, Jennifer Connelly...  United States   
4  Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...  United States   

        Release_Date Rating   Duration  \
0    August 14, 2020  TV-MA  4 Seasons   
1  December 23, 2016  TV-MA     93 min   
2  December 20, 2018      R     78 min   
3  November 16, 2017  PG-13     80 min   
4    January 1, 2020  PG-13    123 min   

                               

In [5]:
validator = BaseValidator(messy_Data)
print(validator.base_validator())

Number of duplicate rows: 2
Percentage of null values in each column:
Show_Id          0.000000
Category         0.000000
Title            0.000000
Director        30.658621
Cast             9.218128
Country          6.509180
Release_Date     0.128386
Rating           0.089870
Duration         0.000000
Type             0.000000
Description      0.000000
dtype: float64
None


In [6]:
#convert headers to lowercase and replace spaces with underscores
def standardize_column_names(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df
current_data = standardize_column_names(messy_Data)

In [7]:
col_header = current_data.columns
print(col_header)

Index(['show_id', 'category', 'title', 'director', 'cast', 'country',
       'release_date', 'rating', 'duration', 'type', 'description'],
      dtype='str')


In [8]:
current_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7789 entries, 0 to 7788
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       7789 non-null   str  
 1   category      7789 non-null   str  
 2   title         7789 non-null   str  
 3   director      5401 non-null   str  
 4   cast          7071 non-null   str  
 5   country       7282 non-null   str  
 6   release_date  7779 non-null   str  
 7   rating        7782 non-null   str  
 8   duration      7789 non-null   str  
 9   type          7789 non-null   str  
 10  description   7789 non-null   str  
dtypes: str(11)
memory usage: 669.5 KB


In [ ]:
#standardizing the data

standardizer = BaseStandardizer(current_data)
standardized_data = standardizer.standardize()

ModuleNotFoundError: No module named 'standardizer'

In [ ]:
user_duplicate_input = input("Do you want to remove duplicates from the data? (yes/no): ").strip().lower()
if user_duplicate_input == 'yes':
    cleaner = DataCleaner(messy_Data)
    duplicates_removed = cleaner.remove_duplicates()
    print("Duplicates have been removed.")
else:
    print("Duplicates cleaning skipped.")

In [ ]:
#replace null values with mean for numeric columns
def replace_null_with_mean(df, columns):
    for col in columns:
        mean_value = df[col].mean()
        df[col].fillna(mean_value, inplace=True)
    return df

In [ ]:
#replace null values with mode for categorical columns
def replace_null_with_mode(df, columns):
    for col in columns:
        mode_value = df[col].mode()[0]
        df[col].fillna(mode_value, inplace=True)
    return df

In [ ]:
user_null_value_input = input("Do you want to handle null values in the data? (yes/no): ").strip().lower()
if user_null_value_input == 'yes':
    numeric_columns = current_data.select_dtypes(include=['number']).columns
    categorical_columns = current_data.select_dtypes(include=['object']).columns

    cleaner = DataCleaner(current_data)
    current_data = cleaner.replace_null_with_mean(current_data, numeric_columns)
    current_data = cleaner.replace_null_with_mode(current_data, categorical_columns)

    print("Null values have been handled.")
else:
    print("Null values handling skipped.")


In [ ]:
#convert datetime columns to datetime format
def convert_datetime_columns(df, columns):
    for col in columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

datetime_columns = ['release_date']
cleaned_data = convert_datetime_columns(cleaned_data, datetime_columns)

In [ ]:
cleaned_data.info()

In [ ]:
#convert to numeric
def convert_numeric_columns(df, columns):
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df



In [ ]:
#remove negative values
def remove_negative_values(df, columns):
    for col in columns:
        df = df[df[col] >= 0]
    return df


In [ ]:
#remove white spaces
def remove_whitespace(df, columns):
    for col in columns:
        df[col] = df[col].str.strip()
    return df

In [ ]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine('sqlite:///cleaned_data.db')
cleaned_data.to_sql('cleaned_netflix_data', con=engine, if_exists='replace', index=False)
print("Cleaned data has been saved to the database.")

In [ ]:
from sqlalchemy import text
with engine.connect() as connection:
    result = connection.execute(text("SELECT * FROM cleaned_netflix_data lIMIT 5"))
    for row in result:
        print(row)